In [ ]:
import pyspark.sql.functions as F
from pyspark.sql import Window
from delta.tables import DeltaTable

In [ ]:
bronze_path = "tihim_project.bronze.products"
silver_path = "tihim_project.silver.products"
quarantine_path = "tihim_project.silver.products_quarantine"
checkpoint_path = "/Volumes/tihim_project/ops/stream_state/checkpoints/silver/products"

In [ ]:
spark.sql(f"""
          
    CREATE TABLE IF NOT EXISTS {silver_path} (
        product_id STRING,
        product_name STRING,
        category STRING,
        brand STRING,
        brand_tier STRING,
        price DOUBLE,
        price_tier STRING,
        ingestion_date TIMESTAMP,
        create_date TIMESTAMP,
        update_date TIMESTAMP
    )
    USING DELTA
    TBLPROPERTIES (
        delta.enableChangeDataFeed = true
    )
""")

In [ ]:
def clean_products(df):
    return (df.drop("_rescued_data", "source_file")
        .withColumn("product_name", F.initcap(F.trim(F.col("product_name"))))
        .withColumn("category", F.initcap(F.trim(F.col("category"))))
        .withColumn("brand", F.upper(F.trim(F.col("brand"))))
        .withColumn("price", F.round(F.col("price"), 2))
        .withColumn("price_tier", F.when(F.col("price") > 1000, "Expensive")
                    .when(F.col("price") > 500, "Moderate")
                    .when(F.col("price") > 100, "Affordable")
                    .otherwise("Budget"))
        .withColumn("brand_tier", F.when(F.upper(F.col("brand")).isin("APPLE", "SONY"), "Prestige")
                    .when(F.col("brand").isin("NIKE", "ADIDAS", "SAMSUNG", "LG"), "Premium")
                    .when(F.col("brand").isin("DELL", "LENOVO", "MAYBELLINE", "REVLON"), "Mainstream")
                    .otherwise("Value"))
    ) 

In [ ]:
def flag_products(df):
    reason = F.concat_ws("; ", F.when(F.col("product_id").isNull(), F.lit("null product_id"))
                         .when(F.col("price").isNull(), F.lit("null price"))
                         .when(F.col("price") < 0, F.lit("negative price"))
                         .when(F.col("product_name").isNull(), F.lit("null product_name")))
    return df.withColumn("dq_reason", reason)



In [ ]:
def upsert_products_to_silver(microBatchId, batchId):

    flagged = flag_products(clean_products(microBatchId))

    valid = flagged.filter(F.col("dq_reason") == "")
    rejects = flagged.filter(F.col("dq_reason") != "")

    if not rejects.isEmpty():
        (
            rejects.withColumn("batch_id", F.lit(batchId))
            .withColumn("rejected_at", F.current_timestamp())
            .write.format("delta").mode("append")
            .option("mergeSchema", "true").saveAsTable(quarantine_path)      
        )

    w = Window.partitionBy("product_id").orderBy(F.col("ingestion_date").desc())
    final = (
        valid.withColumn("rn", F.row_number().over(w))
            .filter(F.col("rn") == 1)
            .drop("rn", "dq_reason")
            .withColumn("create_date", F.col("ingestion_date"))
            .withColumn("update_date", F.col("ingestion_date"))
        )

    try:
        DeltaTable.forName(spark, silver_path).alias("target").merge(
            source = final.alias("update"),
            condition = "target.product_id = update.product_id"

        ).whenMatchedUpdate(
            condition =
                """
                    NOT (target.product_name <=> update.product_name) OR
                    NOT (target.category     <=> update.category)     OR
                    NOT (target.brand        <=> update.brand)        OR
                    NOT (target.brand_tier   <=> update.brand_tier)   OR
                    NOT (target.price        <=> update.price)        OR
                    NOT (target.price_tier   <=> update.price_tier)
                """,

            set = {
                    "product_name": "update.product_name",
                    "category": "update.category",
                    "brand": "update.brand",
                    "brand_tier": "update.brand_tier",
                    "price": "update.price",
                    "price_tier": "update.price_tier",
                    "update_date": "update.update_date"
            }


        ).whenNotMatchedInsert(
            values = {

                "product_id": "update.product_id",
                "product_name": "update.product_name",
                "category": "update.category",
                "brand": "update.brand",
                "brand_tier": "update.brand_tier",
                "price": "update.price",
                "price_tier": "update.price_tier",
                "ingestion_date": "update.ingestion_date",
                "create_date": "update.create_date",
                "update_date": "update.update_date"   

            }
        ).execute()

    except Exception as e:
        print(f"[silver_products] batch {batchId} failed: {e}")
        raise

    print(f"[silver_products] batch {batchId}: {final.count()} valid, {rejects.count()} quarantined")


In [ ]:
query = (
    spark.readStream.table(bronze_path)
    .writeStream.foreachBatch(upsert_products_to_silver)
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .start()
    )

query.awaitTermination()